# 08 — Context Engineering

## Scenario
Northstar has a bot that automatically approves or denies user requests based on the internal policy.

**The Danger:** If we just dump data into the prompt without structure or boundaries, a malicious user can "poison" the context by hiding instructions inside their data. This is known as **Prompt Injection**.

## Step 1: The "Data Dump" (Vulnerable Baseline)

Watch what happens when we just smash the policy and the user data together in a single string.

## Step 2: Context Engineering with Delimiters

We must teach the model the difference between "our instructions" and "untrusted user data". 
The industry standard is to use **XML tags** to create strict boundaries within the context window.

## Conclusion

Context Engineering isn't just about making the prompt shorter; it's about organizing the information. By strictly delimiting system policy from untrusted user inputs, we vastly improve the security and reliability of the application.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab08 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The "Data Dump" (Vulnerable Baseline)

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i08/naive/injection-20-nodes")
print("SYSTEM:\n", request.system)
print("USER (untrusted):\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
assert response.text == "YES"

## Step 2: Context engineering with a policy boundary

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i08/engineered/compliant-3-nodes")
print("SYSTEM:\n", request.system)
print("USER (untrusted):\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
print("PARSED:", response.text)
assert decide(response.text, requested_nodes(CASES[1]["user_data"])) == "approved"
assert instruction_like_score(USER_DATA) >= 0.8

## Step 3: Budget and application control

In [ ]:
sections = [("policy", "At most 5 nodes may be requested.", 100), ("customer request", USER_DATA, 50), ("chat history", "Old conversation", 1)]
packet = build_packet(sections, budget_tokens=15)
print(packet)
assert "[policy]" in packet
assert "[chat history]" not in packet
request = next(r for r in build_requests() if r.case_id == "i08/engineered/polite-7-nodes")
response = client.generate(request)
assert decide(response.text, requested_nodes(CASES[2]["user_data"])) == "rejected"

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.